In [0]:
%sql

SELECT 
description AS appt_desciption, COUNT(DISTINCT sch_appt_id) AS appt_count
FROM 4_prod.raw.mill_sch_appt AS sa
WHERE
description LIKE '%CARDIO%'
GROUP BY description
ORDER BY appt_count DESC

In [0]:
%sql

SELECT Clinic, COUNT(DISTINCT sch_appt_id) AS Count, Tag, COUNT(DISTINCT en.person_id), COUNT(en.person_id)
FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
INNER JOIN 4_prod.raw.mill_sch_appt AS sa
ON fc.clinic = sa.description
LEFT JOIN 4_prod.raw.mill_sch_event AS se
ON sa.sch_event_id = se.sch_event_id
LEFT JOIN 4_prod.raw.mill_encounter AS en
ON se.encntr_id = en.encntr_id
GROUP BY clinic, tag
ORDER BY tag, COUNT(sch_appt_id) DESC

In [0]:
%sql
SELECT * FROM 4_prod.raw.mill_encounter WHERE encntr_id = 0

In [0]:
%sql
WITH a1 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON sa.encntr_id = en.encntr_id
    WHERE CAST(sa.encntr_id AS STRING) != '0'
),
a2 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_sch_event AS se
    ON sa.sch_event_id = se.sch_event_id
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON se.encntr_id = en.encntr_id
    WHERE CAST(se.encntr_id AS STRING) != '0'
),
u AS (
    SELECT *
    FROM a1
    UNION
    SELECT *
    FROM a2
)
SELECT Tag, COUNT(DISTINCT person_id) AS PatientCount
FROM u
GROUP BY Tag

In [0]:
%sql
CREATE TABLE 5_projects.dar064.rde_patient_demographics AS
WITH a1 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON sa.encntr_id = en.encntr_id
    WHERE CAST(sa.encntr_id AS STRING) != '0'
),
a2 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_sch_event AS se
    ON sa.sch_event_id = se.sch_event_id
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON se.encntr_id = en.encntr_id
    WHERE CAST(se.encntr_id AS STRING) != '0'
),
u AS (
    SELECT DISTINCT person_id
    FROM a1
    UNION
    SELECT DISTINCT person_id
    FROM a2
)
SELECT pd.* EXCEPT (NHS_Number, MRN, Date_of_Birth)
FROM u
INNER JOIN 4_prod.rde.rde_patient_demographics AS pd
ON u.PERSON_ID = pd.PERSON_ID


In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT PERSON_ID) FROM 5_projects.dar064.rde_patient_demographics

In [0]:
from tqdm.notebook import tqdm

table_names = ["rde_apc_diagnosis", "rde_apc_opcs", "rde_ariapharmacy", "rde_blobdataset", "rde_cds_apc", "rde_cds_opa", "rde_emergency_findings", "rde_emergencyd", "rde_encounter", "rde_measurements", "rde_medadmin", "rde_op_diagnosis", "rde_opa_opcs", "rde_pathology", "rde_pharmacyorders", "rde_powerforms", "rde_radiology", "rde_raw_pathology"]

for table_name in tqdm(table_names):
    q = f"""CREATE OR REPLACE TABLE 5_projects.dar064.{table_name} AS
    SELECT t.*
    FROM 4_prod.rde.{table_name} AS t
    INNER JOIN 5_projects.dar064.rde_patient_demographics AS pd
    ON t.person_id = pd.person_id"""
    spark.sql(q)

In [0]:
from pyspark.sql.utils import AnalysisException

for table_name in tqdm(table_names):
    df = spark.table(f"5_projects.dar064.{table_name}")
    cols_to_drop = [col for col in ["NHS_Number", "MRN", "BlobContents"] if col in df.columns]
    if cols_to_drop:
        df = df.drop(*cols_to_drop)
        df.write.mode("overwrite").saveAsTable(f"5_projects.dar064.{table_name}")

In [0]:
%sql
SELECT COUNT(*) FROM 5_projects.dar064.rde_blobdataset

In [0]:
%sql
WITH a1 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id, COALESCE(YEAR(sa.BEG_DT_TM), YEAR(sa.END_DT_TM)) AS appt_year
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON sa.encntr_id = en.encntr_id
    WHERE CAST(sa.encntr_id AS STRING) != '0'
),
a2 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id, COALESCE(YEAR(sa.BEG_DT_TM), YEAR(sa.END_DT_TM)) AS appt_year
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_sch_event AS se
    ON sa.sch_event_id = se.sch_event_id
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON se.encntr_id = en.encntr_id
    WHERE CAST(se.encntr_id AS STRING) != '0'
),
u AS (
    SELECT *
    FROM a1
    UNION
    SELECT *
    FROM a2
)
SELECT Tag, appt_year, COUNT(DISTINCT sch_appt_id) AS appt_count
FROM u
GROUP BY Tag, appt_year
ORDER BY Tag, appt_year

In [0]:
%sql
WITH a1 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id, sa.BEG_DT_TM
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON sa.encntr_id = en.encntr_id
    WHERE CAST(sa.encntr_id AS STRING) != '0'
),
a2 AS (
    SELECT idx, Clinic, sch_appt_id, Tag, en.person_id, sa.BEG_DT_TM
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_sch_event AS se
    ON sa.sch_event_id = se.sch_event_id
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON se.encntr_id = en.encntr_id
    WHERE CAST(se.encntr_id AS STRING) != '0'
),
u AS (
    SELECT *
    FROM a1
    UNION
    SELECT *
    FROM a2
)
SELECT Tag, MIN(DATE(BEG_DT_TM)) AS min_appt_date, MAX(DATE(BEG_DT_TM)) AS max_appt_date
FROM u
GROUP BY Tag
ORDER BY Tag

In [0]:
%sql
CREATE TABLE 5_projects.dar064.filtered_clinics_appt AS
WITH a1 AS (
    SELECT Clinic, sch_appt_id, Tag, en.person_id, sa.BEG_DT_TM, sa.END_DT_TM, sa.ACTIVE_IND
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON sa.encntr_id = en.encntr_id
    WHERE CAST(sa.encntr_id AS STRING) != '0'
),
a2 AS (
    SELECT Clinic, sch_appt_id, Tag, en.person_id, sa.BEG_DT_TM, sa.END_DT_TM, sa.ACTIVE_IND
    FROM 5_projects.dar064.filtered_clinics_20260612 AS fc
    INNER JOIN 4_prod.raw.mill_sch_appt AS sa
    ON fc.clinic = sa.description
    LEFT JOIN 4_prod.raw.mill_sch_event AS se
    ON sa.sch_event_id = se.sch_event_id
    LEFT JOIN 4_prod.raw.mill_encounter AS en
    ON se.encntr_id = en.encntr_id
    WHERE CAST(se.encntr_id AS STRING) != '0'
)
SELECT *
FROM a1
UNION
SELECT *
FROM a2

In [0]:
%sql
SELECT *
FROM 4_prod.pacs_dlt.pacs_examcode_dict
WHERE 
(
    system = 'Structure of cardiovascular system (body structure)'
    AND region = 'Thoracic structure (body structure)'
    AND sub_modality_id IS NULL
    AND modality_id = 113091000
)
LIMIT 100

In [0]:
%sql
WITH ec AS (
    SELECT short_code
    FROM 4_prod.pacs_dlt.pacs_examcode_dict
    WHERE 
        system = 'Structure of cardiovascular system (body structure)'
        AND region = 'Thoracic structure (body structure)'
        AND sub_modality_id IS NULL
        AND modality_id = 113091000
)
SELECT examcode, MODE(ExaminationDescription) AS description, COUNT(DISTINCT Clinical_Event_Id) AS count FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN 5_projects.dar064.rde_patient_demographics AS pd
ON im.personid = pd.person_id
INNER JOIN ec
ON im.examcode = ec.short_code
GROUP BY examcode
ORDER BY count DESC

In [0]:
%sql
SELECT examcode, MODE(ExaminationDescription) AS description, COUNT(DISTINCT Clinical_Event_Id) AS count FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN 5_projects.dar064.rde_patient_demographics AS pd
ON im.personid = pd.person_id
WHERE examcode IN ('MCARD', 'MCSPS')
GROUP BY examcode
ORDER BY count DESC

In [0]:
%sql
SELECT AccessionNbr FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN 5_projects.dar064.rde_patient_demographics AS pd
ON im.personid = pd.person_id
WHERE examcode IN ('MCARD')
LIMIT 100

In [0]:
%sql
SELECT AccessionNbr FROM 4_prod.pacs.imaging_metadata AS im
INNER JOIN 5_projects.dar064.rde_patient_demographics AS pd
ON im.personid = pd.person_id
WHERE examcode IN ('MCSPS')
LIMIT 100

In [0]:
tables = [row.tableName for row in spark.sql("SHOW TABLES IN 5_projects.dar064").collect()]
for table in tables:
    view_name = f"{table}_view"
    spark.sql(f"CREATE OR REPLACE VIEW 5_projects.dar064.{view_name} AS SELECT * FROM 5_projects.dar064.{table}")

In [0]:
%sql
SELECT * FROM 5_projects.dar064.rde_radiology
WHERE PERSON_ID = 5599387
LIMIT 100

In [0]:
%sql
SELECT * FROM 5_projects.dar064.rde_blobdataset
WHERE PERSON_ID = 5599387
LIMIT 100

In [0]:
%sql
CREATE OR REPLACE TABLE 5_projects.dar064.imaging_report AS
SELECT ir.* EXCEPT(Mrn, NHs_Number, BlobContents, AnonymizedText), bd.anonymizedtext
--SELECT COUNT(distinct EVENTID)
FROM 4_prod.pacs.imaging_report AS ir
INNER JOIN 5_projects.dar064.rde_patient_demographics AS pd
ON ir.personid = pd.person_id
LEFT JOIN 4_prod.rde.rde_blobdataset AS bd
ON ir.ReportEventId = bd.EventId
WHERE ExamCode IN ('MCARD', 'MCSPS', 'UTECG', 'UTE3D')

In [0]:
%sql

--DROP VIEW 5_projects.dar064.imaging_report_view
CREATE VIEW 5_projects.dar064.imaging_report_view AS
SELECT * FROM 5_projects.dar064.imaging_report

In [0]:
%sql
SELECT * FROM 4_prod.pacs_dlt.pacs_examcode_dict
WHERE modality_id = 16310003 -- US
        AND system = 'Structure of cardiovascular system (body structure)'
        AND region = 'Thoracic structure (body structure)'